# Vision AI Studio — exploration notebookOriginal Colab exploration that the production app in `src/vision_ai_studio` was extracted from.- Cells below use Colab (`google.colab.drive`) and local Drive paths. Update `IMG_DIR`/`CAPTIONS_JSON` before running.- The Streamlit prototype, ngrok tunnel and hard-coded credentials were removed from this notebook.- The maintained application lives in `app/` and `src/`; this file is kept as a record of the experiment.Dataset: [COCO 2017](https://cocodataset.org/) (val2017 + annotations), not committed.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

IMG_DIR = "/content/drive/MyDrive/COCO DATASET (1)/val2017/val2017"
CAPTIONS_JSON = "/content/drive/MyDrive/COCO DATASET (1)/annotations/captions_val2017.json/captions_val2017.json"
INSTANCES_JSON = "/content/drive/MyDrive/COCO DATASET (1)/annotations/captions_val2017.json/instances_val2017.json/instances_val2017.json"

In [ ]:
import os
import random

import cv2
import matplotlib.pyplot as plt
from PIL import Image
from pycocotools.coco import COCO

coco_caps = COCO(CAPTIONS_JSON)

img_id = random.choice(coco_caps.getImgIds())

img_info = coco_caps.loadImgs(img_id)[0]
img_path = os.path.join(IMG_DIR, img_info["file_name"])

img = cv2.imread(img_path)
if img is None:
    raise FileNotFoundError(f"Image not found: {img_path}")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(img)
plt.axis("off")
plt.title("Original Image")
plt.show()

ann_ids = coco_caps.getAnnIds(imgIds=img_id)
anns = coco_caps.loadAnns(ann_ids)
print(f"📂 Image ID: {img_id}")
print("📜 Human Captions:")
for i, ann in enumerate(anns):
    print(f"{i + 1}. {ann['caption']}")

In [ ]:
import torch
from transformers import BlipForConditionalGeneration, BlipProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
caption_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(device)

In [ ]:
from torchvision import models, transforms
from torchvision.models.segmentation import DeepLabV3_ResNet50_Weights

weights = DeepLabV3_ResNet50_Weights.DEFAULT
seg_model = models.segmentation.deeplabv3_resnet50(weights=weights).eval()

In [ ]:
VOC_LABELS = {
    0: "background",
    1: "aeroplane",
    2: "bicycle",
    3: "bird",
    4: "boat",
    5: "bottle",
    6: "bus",
    7: "car",
    8: "cat",
    9: "chair",
    10: "cow",
    11: "diningtable",
    12: "dog",
    13: "horse",
    14: "motorbike",
    15: "person",
    16: "potted plant",
    17: "sheep",
    18: "sofa",
    19: "train",
    20: "tvmonitor",
}

In [ ]:
def analyze_image(img_path, coco_caps, caption_model, caption_processor, seg_model, voc_labels):
    import cv2
    import matplotlib.pyplot as plt
    import numpy as np
    import seaborn as sns
    import torch

    # Load and convert image
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {img_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img).convert("RGB")

    # Generate Caption
    inputs = caption_processor(images=pil_img, return_tensors="pt").to(caption_model.device)
    output = caption_model.generate(**inputs)
    caption = caption_processor.decode(output[0], skip_special_tokens=True)

    # Segment Image
    transform = transforms.Compose(
        [
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )
    input_tensor = transform(pil_img).unsqueeze(0)
    with torch.no_grad():
        seg_output = seg_model(input_tensor)["out"][0]
    segmentation = seg_output.argmax(0)

    # Show Image + Segmentation
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    plt.imshow(pil_img)
    plt.axis("off")
    plt.title("Original Image")
    plt.subplot(1, 2, 2)
    plt.imshow(segmentation.cpu(), cmap="tab20b")
    plt.axis("off")
    plt.title("Segmentation Mask")
    plt.suptitle(f"📜 Caption: {caption}", fontsize=14)
    plt.tight_layout()
    plt.show()

    # Pixel Count Bar Plot
    unique_classes, counts = np.unique(segmentation.numpy(), return_counts=True)
    plt.figure(figsize=(12, 4))
    sns.barplot(x=unique_classes, y=counts, palette="viridis")
    plt.xlabel("Class Label ID")
    plt.ylabel("Pixel Count")
    plt.title("Detected Segmentation Classes (by pixel area)")
    plt.show()

    # Print Labels
    print("🧠 Classes present in the image:")
    for cls_id in unique_classes:
        if cls_id in voc_labels:
            pixel_count = counts[np.where(unique_classes == cls_id)][0]
            print(f" - {voc_labels[cls_id]} (Label {cls_id}, Pixels: {pixel_count})")

    print(
        "🎯 Top object in segmentation:",
        voc_labels.get(int(unique_classes[np.argmax(counts)]), "unknown"),
    )
    print("🔍 Caption from BLIP:", caption)

    return caption

In [ ]:
# Call the function
caption_result = analyze_image(
    img_path=img_path,
    coco_caps=coco_caps,
    caption_model=caption_model,
    caption_processor=processor,
    seg_model=seg_model,
    voc_labels=VOC_LABELS,
)

In [ ]:
img_ids = coco_caps.getImgIds()
for img_id in random.sample(img_ids, 3):  # Pick 3 random images
    img_info = coco_caps.loadImgs(img_id)[0]
    path = os.path.join(IMG_DIR, img_info["file_name"])
    print(f"\n🔎 Analyzing Image ID: {img_id}")
    analyze_image(path, coco_caps, caption_model, processor, seg_model, VOC_LABELS)